In [17]:
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier

DATA_DIR = Path("../data/raw")
TRAIN_TRANSACTION_PATH = DATA_DIR / "train_transaction.csv"
TRAIN_IDENTITY_PATH = DATA_DIR / "train_identity.csv"
OUTPUT_METRICS_DIR = Path("../outputs/metrics")
PROCESSED_DATA_DIR = Path("../data/processed")

TARGET_COL = "isFraud"

RANDOM_STATE = 42

In [18]:
transaction = pd.read_csv(TRAIN_TRANSACTION_PATH)
identity = pd.read_csv(TRAIN_IDENTITY_PATH)

df = transaction.merge(
    identity,
    on="TransactionID",
    how="left"
)

print("Transaction shape:", transaction.shape)
print("Identity shape:", identity.shape)
print("Merged shape:", df.shape)
print("Merge satir sayisi dogru mu:", df.shape[0] == transaction.shape[0])

Transaction shape: (590540, 394)
Identity shape: (144233, 41)
Merged shape: (590540, 434)
Merge satir sayisi dogru mu: True


In [19]:
X = df.drop(columns=[TARGET_COL, "TransactionID"], errors="ignore")
y = df[TARGET_COL].astype(int)

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.40,
    random_state=RANDOM_STATE,
    stratify=y
)

X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=RANDOM_STATE,
    stratify=y_temp
)

print("Train shape:", X_train.shape, y_train.shape)
print("Validation shape:", X_valid.shape, y_valid.shape)
print("Test shape:", X_test.shape, y_test.shape)

print("\nFraud oranları:")
print("Train:", y_train.mean())
print("Validation:", y_valid.mean())
print("Test:", y_test.mean())


Train shape: (354324, 432) (354324,)
Validation shape: (118108, 432) (118108,)
Test shape: (118108, 432) (118108,)

Fraud oranları:
Train: 0.03499057359930459
Validation: 0.0349933958749619
Test: 0.03498492904798998


In [20]:
numeric_features = X_train.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X_train.select_dtypes(include=["object", "str", "category", "bool"]).columns.tolist()

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=20))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

print("Sayisal sutun sayisi:", len(numeric_features))
print("Kategorik sutun sayisi:", len(categorical_features))
print("Toplam ozellik sayisi:", len(numeric_features) + len(categorical_features))

Sayisal sutun sayisi: 401
Kategorik sutun sayisi: 31
Toplam ozellik sayisi: 432


In [21]:
import time

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

In [22]:
def evaluate_thresholds(model_name, y_true, y_prob):
    rows = []

    roc_auc = roc_auc_score(y_true, y_prob)
    pr_auc = average_precision_score(y_true, y_prob)

    for threshold in np.arange(0.05, 0.55, 0.05):
        threshold = round(float(threshold), 2)
        y_pred = (y_prob >= threshold).astype(int)

        tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

        rows.append({
            "model": model_name,
            "threshold": threshold,
            "precision": precision_score(y_true, y_pred, zero_division=0),
            "recall": recall_score(y_true, y_pred, zero_division=0),
            "f1": f1_score(y_true, y_pred, zero_division=0),
            "roc_auc": roc_auc,
            "pr_auc": pr_auc,
            "routed_rate": y_pred.mean(),
            "tn": tn,
            "fp": fp,
            "fn": fn,
            "tp": tp
        })

    return pd.DataFrame(rows)

In [23]:
def measure_inference_time(model, X_data, sample_size=10000, repeat_count=5):
    sample = X_data.iloc[:min(sample_size, len(X_data))]

    times = []

    for _ in range(repeat_count):
        start = time.perf_counter()
        _ = model.predict_proba(sample)[:, 1]
        end = time.perf_counter()

        times.append(end - start)

    return {
        "sample_size": len(sample),
        "inference_mean_sec": np.mean(times),
        "inference_std_sec": np.std(times)
    }

In [24]:
small_rf_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=30,
        max_depth=8,
        min_samples_leaf=50,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])

small_rf_model.fit(X_train, y_train)

small_rf_val_prob = small_rf_model.predict_proba(X_valid)[:, 1]

small_rf_threshold_results = evaluate_thresholds(
    model_name="Kucuk Random Forest",
    y_true=y_valid,
    y_prob=small_rf_val_prob
)

small_rf_threshold_results

,model,threshold,precision,recall,f1,roc_auc,pr_auc,routed_rate,tn,fp,fn,tp
0,Kucuk Random Forest,0.05,0.035060,1.000000,0.067744,0.863875,0.437393,0.998112,223,113752,0,4133
1,Kucuk Random Forest,0.10,0.035072,1.000000,0.067768,0.863875,0.437393,0.997748,266,113709,0,4133
2,Kucuk Random Forest,0.15,0.041055,0.987660,0.078834,0.863875,0.437393,0.841831,18630,95345,51,4082
3,Kucuk Random Forest,0.20,0.046737,0.962255,0.089144,0.863875,0.437393,0.720468,32859,81116,156,3977
4,Kucuk Random Forest,0.25,0.048550,0.950883,0.092383,0.863875,0.437393,0.685373,36957,77018,203,3930
5,Kucuk Random Forest,0.30,0.057874,0.928623,0.108958,0.863875,0.437393,0.561486,51497,62478,295,3838
6,Kucuk Random Forest,0.35,0.083066,0.857730,0.151463,0.863875,0.437393,0.361339,74843,39132,588,3545
7,Kucuk Random Forest,0.40,0.097196,0.821921,0.173835,0.863875,0.437393,0.295916,82422,31553,736,3397
8,Kucuk Random Forest,0.45,0.116687,0.773046,0.202767,0.863875,0.437393,0.231830,89789,24186,938,3195
9,Kucuk Random Forest,0.50,0.143013,0.729736,0.239156,0.863875,0.437393,0.178557,95902,18073,1117,3016


In [25]:
small_rf_inference_time = measure_inference_time(
    model=small_rf_model,
    X_data=X_valid,
    sample_size=10000,
    repeat_count=5
)

small_rf_inference_time

{'sample_size': 10000,
 'inference_mean_sec': np.float64(0.3140535800001089),
 'inference_std_sec': np.float64(0.009772846010225496)}

In [26]:
rf_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=150,
        max_depth=None,
        min_samples_leaf=10,
        class_weight="balanced_subsample",
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])

rf_model.fit(X_train, y_train)

rf_val_prob = rf_model.predict_proba(X_valid)[:, 1]

rf_threshold_results = evaluate_thresholds(
    model_name="Random Forest",
    y_true=y_valid,
    y_prob=rf_val_prob
)

rf_threshold_results

,model,threshold,precision,recall,f1,roc_auc,pr_auc,routed_rate,tn,fp,fn,tp
0,Random Forest,0.05,0.044834,0.993225,0.085796,0.930379,0.622851,0.775214,26521,87454,28,4105
1,Random Forest,0.10,0.062909,0.971207,0.118165,0.930379,0.622851,0.540234,54183,59792,119,4014
2,Random Forest,0.15,0.088387,0.943624,0.161635,0.930379,0.622851,0.373590,73751,40224,233,3900
3,Random Forest,0.20,0.122005,0.904428,0.215007,0.930379,0.622851,0.259407,87075,26900,395,3738
4,Random Forest,0.25,0.161535,0.865715,0.272267,0.930379,0.622851,0.187540,95403,18572,555,3578
5,Random Forest,0.30,0.205866,0.821921,0.329262,0.930379,0.622851,0.139711,100871,13104,736,3397
6,Random Forest,0.35,0.260491,0.774982,0.389920,0.930379,0.622851,0.104108,104882,9093,930,3203
7,Random Forest,0.40,0.325632,0.729736,0.450317,0.930379,0.622851,0.078420,107729,6246,1117,3016
8,Random Forest,0.45,0.397599,0.681103,0.502096,0.930379,0.622851,0.059945,109710,4265,1318,2815
9,Random Forest,0.50,0.477079,0.637068,0.545586,0.930379,0.622851,0.046728,111089,2886,1500,2633


In [27]:
rf_inference_time = measure_inference_time(
    model=rf_model,
    X_data=X_valid,
    sample_size=10000,
    repeat_count=5
)

rf_inference_time

{'sample_size': 10000,
 'inference_mean_sec': np.float64(0.43171625999966634),
 'inference_std_sec': np.float64(0.007752875298561855)}

In [28]:
selected_threshold = 0.20
selected_model_name = "Random Forest"

selected_test_prob = rf_model.predict_proba(X_test)[:, 1]
selected_test_pred = (selected_test_prob >= selected_threshold).astype(int)

test_confusion = confusion_matrix(y_test, selected_test_pred)
tn, fp, fn, tp = test_confusion.ravel()

test_metrics = {
    "model": selected_model_name,
    "threshold": selected_threshold,
    "precision": precision_score(y_test, selected_test_pred, zero_division=0),
    "recall": recall_score(y_test, selected_test_pred, zero_division=0),
    "f1": f1_score(y_test, selected_test_pred, zero_division=0),
    "roc_auc": roc_auc_score(y_test, selected_test_prob),
    "pr_auc": average_precision_score(y_test, selected_test_prob),
    "routed_rate": selected_test_pred.mean(),
    "tn": tn,
    "fp": fp,
    "fn": fn,
    "tp": tp,
}

pd.DataFrame([test_metrics])

,model,threshold,precision,recall,f1,roc_auc,pr_auc,routed_rate,tn,fp,fn,tp
0,Random Forest,0.2,0.121243,0.901985,0.213753,0.927205,0.610005,0.26027,86963,27013,405,3727


In [29]:
selected_threshold = 0.20
selected_model_name = "Random Forest"

selected_test_prob = rf_model.predict_proba(X_test)[:, 1]
selected_test_pred = (selected_test_prob >= selected_threshold).astype(int)

test_confusion = confusion_matrix(y_test, selected_test_pred)
tn, fp, fn, tp = test_confusion.ravel()

test_metrics = {
    "model": selected_model_name,
    "threshold": selected_threshold,
    "precision": precision_score(y_test, selected_test_pred, zero_division=0),
    "recall": recall_score(y_test, selected_test_pred, zero_division=0),
    "f1": f1_score(y_test, selected_test_pred, zero_division=0),
    "roc_auc": roc_auc_score(y_test, selected_test_prob),
    "pr_auc": average_precision_score(y_test, selected_test_prob),
    "routed_rate": selected_test_pred.mean(),
    "tn": tn,
    "fp": fp,
    "fn": fn,
    "tp": tp,
}

pd.DataFrame([test_metrics])

,model,threshold,precision,recall,f1,roc_auc,pr_auc,routed_rate,tn,fp,fn,tp
0,Random Forest,0.2,0.121243,0.901985,0.213753,0.927205,0.610005,0.26027,86963,27013,405,3727


In [30]:
OUTPUT_METRICS_DIR.mkdir(parents=True, exist_ok=True)

small_rf_threshold_results["inference_mean_sec"] = small_rf_inference_time["inference_mean_sec"]
small_rf_threshold_results["inference_std_sec"] = small_rf_inference_time["inference_std_sec"]

rf_threshold_results["inference_mean_sec"] = rf_inference_time["inference_mean_sec"]
rf_threshold_results["inference_std_sec"] = rf_inference_time["inference_std_sec"]

phase2_results = pd.concat(
    [small_rf_threshold_results, rf_threshold_results],
    ignore_index=True
)

phase2_results_path = OUTPUT_METRICS_DIR / "phase2_random_forest_threshold_results.csv"
test_metrics_path = OUTPUT_METRICS_DIR / "phase2_selected_test_metrics.csv"

phase2_results.to_csv(phase2_results_path, index=False)
pd.DataFrame([test_metrics]).to_csv(test_metrics_path, index=False)

print("Asama 2 Random Forest sonuclari kaydedildi:")
print(phase2_results_path)
print("Seçilen modelin test metrikleri kaydedildi:")
print(test_metrics_path)

phase2_results

Asama 2 Random Forest sonuclari kaydedildi:
..\outputs\metrics\phase2_random_forest_threshold_results.csv
Seçilen modelin test metrikleri kaydedildi:
..\outputs\metrics\phase2_selected_test_metrics.csv


,model,threshold,precision,recall,f1,roc_auc,pr_auc,routed_rate,tn,fp,fn,tp,inference_mean_sec,inference_std_sec
0,Kucuk Random Forest,0.05,0.035060,1.000000,0.067744,0.863875,0.437393,0.998112,223,113752,0,4133,0.314054,0.009773
1,Kucuk Random Forest,0.10,0.035072,1.000000,0.067768,0.863875,0.437393,0.997748,266,113709,0,4133,0.314054,0.009773
2,Kucuk Random Forest,0.15,0.041055,0.987660,0.078834,0.863875,0.437393,0.841831,18630,95345,51,4082,0.314054,0.009773
3,Kucuk Random Forest,0.20,0.046737,0.962255,0.089144,0.863875,0.437393,0.720468,32859,81116,156,3977,0.314054,0.009773
4,Kucuk Random Forest,0.25,0.048550,0.950883,0.092383,0.863875,0.437393,0.685373,36957,77018,203,3930,0.314054,0.009773
5,Kucuk Random Forest,0.30,0.057874,0.928623,0.108958,0.863875,0.437393,0.561486,51497,62478,295,3838,0.314054,0.009773
6,Kucuk Random Forest,0.35,0.083066,0.857730,0.151463,0.863875,0.437393,0.361339,74843,39132,588,3545,0.314054,0.009773
7,Kucuk Random Forest,0.40,0.097196,0.821921,0.173835,0.863875,0.437393,0.295916,82422,31553,736,3397,0.314054,0.009773
8,Kucuk Random Forest,0.45,0.116687,0.773046,0.202767,0.863875,0.437393,0.231830,89789,24186,938,3195,0.314054,0.009773
9,Kucuk Random Forest,0.50,0.143013,0.729736,0.239156,0.863875,0.437393,0.178557,95902,18073,1117,3016,0.314054,0.009773


In [31]:
import joblib

MODEL_DIR = Path("../models/light")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

selected_model_name = "Random Forest"
selected_threshold = 0.20
selected_model = rf_model

selected_model_metrics = phase2_results[
    (phase2_results["model"] == selected_model_name) &
    (phase2_results["threshold"] == selected_threshold)
].iloc[0].to_dict()

model_bundle = {
    "pipeline": selected_model,
    "threshold": selected_threshold,
    "model_name": selected_model_name,
    "feature_columns": X_train.columns.tolist(),
    "numeric_features": numeric_features,
    "categorical_features": categorical_features,
    "validation_metrics": selected_model_metrics,
    "test_metrics": test_metrics,
    "note": "Test verisi model seciminde kullanilmadi."
}

model_path = MODEL_DIR / "best_light_random_forest_model.joblib"

joblib.dump(model_bundle, model_path)

print("Model, preprocessing ve threshold birlikte kaydedildi:")
print(model_path)

Model, preprocessing ve threshold birlikte kaydedildi:
..\models\light\best_light_random_forest_model.joblib
